# 第5章 LoRA / QLoRA 実践

この章では、事前学習済みの小型 causal LM に LoRA adapter を付け、教材用の入出力ペアで実際に supervised fine-tuning を行います。ゼロから小型言語モデルを作るのではなく、既存モデルの重みを固定し、追加 adapter だけを学習します。

RTX 4060 Ti 16GB で回るように、既定では `Qwen/Qwen2.5-0.5B-Instruct`、短い `max_length`、`batch size 1`、少ない `max_steps` にしています。学習済み adapter は `work/local-llm-training/` に保存され、git には入りません。

QLoRA は Windows の bitsandbytes 対応で詰まることがあるため、この Notebook では LoRA を必須の実行経路にします。量子化を使う QLoRA は、環境が整ったら同じデータと評価で置き換える発展課題として扱います。


In [ ]:
from pathlib import Path
import os
import json
import sys

# Notebook をどこから開いても helper を import できるようにします。
search_roots = [Path.cwd()]
env_root = os.environ.get("LOCAL_LLM_REPO_ROOT")
if env_root:
    search_roots.append(Path(env_root))
search_roots.append(Path("C:/LLM"))

seen = set()
for root in search_roots:
    current = root.resolve()
    for candidate in [current, *current.parents]:
        if candidate in seen:
            continue
        seen.add(candidate)
        helper_dir = candidate / "notebooks"
        if (helper_dir / "local_llm_practice.py").exists():
            sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise RuntimeError(
        "notebooks/local_llm_practice.py が見つかりません。"
        "C:/LLM か notebooks/ 配下で開くか、LOCAL_LLM_REPO_ROOT を設定してください。"
    )

from local_llm_practice import (
    DATA_DIR,
    DOCS_DIR,
    REPO_ROOT,
    WORK_DIR,
    TrainingConfig,
    ask_about_chapter,
    attach_lora,
    configure_local_caches,
    count_trainable_parameters,
    generate_text,
    gpu_summary,
    load_base_model,
    load_chapter,
    load_jsonl,
    make_cpt_features,
    make_sft_features,
    nvidia_smi_summary,
    ollama_generate,
    print_headings,
    read_text,
    retrieve_chunks,
    split_markdown,
    train_lora_adapter,
)

configure_local_caches()
print("REPO_ROOT:", REPO_ROOT)
print("DOCS_DIR :", DOCS_DIR)
print("WORK_DIR :", WORK_DIR)

chapter_path, chapter_text = load_chapter("05-lora-qlora.md")
print(chapter_path)
print_headings(chapter_text)


## 1. GPU と学習設定を確認する

まず PyTorch から RTX 4060 Ti が見えるか確認します。CUDA が使えない場合、この章の実学習は進めず、CUDA 版 PyTorch の導入を先に行います。


In [ ]:
# nvidia-smi はドライバ側、gpu_summary は PyTorch 側の見え方です。
print(nvidia_smi_summary())
print(json.dumps(gpu_summary(), ensure_ascii=False, indent=2))

config = TrainingConfig(
    model_id="Qwen/Qwen2.5-0.5B-Instruct",
    max_length=192,
    max_steps=20,
    learning_rate=2e-4,
    lora_r=8,
    lora_alpha=16,
)
print(config)


## 2. 教材用の入出力ペアを読む

LoRA は知識を丸暗記させるより、出力形式、文体、分類基準を安定させたい時に使います。ここでは公開可否を確認済みの教材用 JSONL だけを使います。


In [ ]:
records = load_jsonl(DATA_DIR / "lora_dummy_dataset.jsonl")
print("records:", len(records))
print(json.dumps(records[0], ensure_ascii=False, indent=2))


## 3. 事前学習済みモデルを読み、学習前の応答を見る

学習前の応答を残しておくと、adapter 学習後に形式がどれくらい寄ったか比較できます。


In [ ]:
# load_base_model は Hugging Face から公開済みモデルを読み、CUDA に載せます。
tokenizer, base_model = load_base_model(config)

prompt, expected = records[0]["instruction"] + "\n" + records[0]["input"], records[0]["output"]
before_prompt = f"""
### 指示
{records[0]['instruction']}

### 入力
{records[0]['input']}

### 応答
""".strip()

before_text = generate_text(base_model, tokenizer, before_prompt, max_new_tokens=96)
print(before_text)


## 4. LoRA adapter を付ける

ここでベースモデル本体は固定され、LoRA の小さな追加行列だけが trainable になります。`trainable ratio` が小さいほど、少ない VRAM で試しやすくなります。


In [ ]:
lora_model = attach_lora(base_model, config)
print(json.dumps(count_trainable_parameters(lora_model), ensure_ascii=False, indent=2))


## 5. LoRA adapter を実際に学習する

`make_sft_features` は、指示と入力をプロンプト、`output` を正解として扱います。プロンプト部分の loss は mask し、応答部分を中心に adapter を更新します。


In [ ]:
features = make_sft_features(records, tokenizer, max_length=config.max_length)
adapter_dir = WORK_DIR / "chapter05-lora-adapter"

metrics = train_lora_adapter(
    lora_model,
    tokenizer,
    features,
    adapter_dir,
    config,
)
print(json.dumps(metrics, ensure_ascii=False, indent=2))


## 6. 学習後の応答と前後比較

loss が下がっても、それだけでは教材としては不十分です。同じプロンプトで、形式や文体が教材データ側へ寄ったかを確認します。


In [ ]:
after_text = generate_text(lora_model, tokenizer, before_prompt, max_new_tokens=96)
print("=== before ===")
print(before_text)
print("\n=== after ===")
print(after_text)
print("\n=== expected style ===")
print(expected)


## 7. 生成物を確認する

adapter は `work/` 配下に保存します。PR や公開 repository へ入れるのは Notebook と教材サンプルだけで、学習済み重みは入れません。


In [ ]:
for path in sorted(adapter_dir.glob("*")):
    print(path)
